# 05 — Waveform preprocessing V2

This pipeline corrects the annotated-channel mismatch in the baseline and
preserves the complete 10-second waveform for temporal modeling.

- `PPG_R` is used because BUT PPG quality labels were created from red PPG.
- Each record becomes three aligned views: robust raw, Butterworth band-pass,
  and first derivative.
- The original subject-disjoint split is retained.
- No label-dependent transformation or overlapping cross-subject windowing is used.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
AI_SERVICE_ROOT = REPO_ROOT / "apps" / "ai-service"
sys.path.insert(0, str(AI_SERVICE_ROOT))

RAW_DIR = Path(
    os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")
).expanduser().resolve()
WAVEFORM_DIR = Path(
    os.environ.get("PURE_WAVEFORM_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-waveforms-v2")
).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PURE_MODEL_DIR", REPO_ROOT / "models")).expanduser().resolve()

print("Repository:", REPO_ROOT)
print("Raw dataset:", RAW_DIR)
print("Waveform dataset:", WAVEFORM_DIR)
print("Model directory:", MODEL_DIR)

In [ ]:
import subprocess

command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "prepare_but_ppg_waveforms.py"),
    "--raw-dir", str(RAW_DIR),
    "--output-dir", str(WAVEFORM_DIR),
]
subprocess.run(command, check=True)

In [ ]:
import json
import numpy as np
import pandas as pd

archive = np.load(WAVEFORM_DIR / "waveforms.npz")
X, y = archive["X"], archive["y"]
metadata = pd.DataFrame({
    "record_id": archive["record_id"],
    "subject_id": archive["subject_id"],
    "split": archive["split"],
    "label_good_quality": y,
    "selected_channel": archive["selected_channel"],
})
print("X shape:", X.shape)
print("finite:", bool(np.isfinite(X).all()))
display(pd.crosstab(metadata["split"], metadata["label_good_quality"], margins=True))
display(pd.crosstab(metadata["selected_channel"], metadata["label_good_quality"], margins=True))
json.loads((WAVEFORM_DIR / "manifest.json").read_text())

In [ ]:
import matplotlib.pyplot as plt

view_names = ["raw robust", "band-pass", "first derivative"]
figure, axes = plt.subplots(2, 3, figsize=(15, 6), sharex=True)
for row, label in enumerate([1, 0]):
    index = int(np.flatnonzero(y == label)[0])
    for column in range(3):
        axes[row, column].plot(X[index, column], linewidth=1)
        axes[row, column].set_title(f"{'Good' if label else 'Poor'} — {view_names[column]}")
plt.tight_layout()